In [1]:
import pandas as pd
import numpy as np
import statsmodels.api as sm
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LassoCV
from sklearn.pipeline import Pipeline

# ============================================================
# STEP 0 — Load raw datasets from your folder
# ============================================================

BASE = "data/macro_processed"

paths = {
    # Returns building blocks
    "sp500":        f"{BASE}/other/sp500_processed.csv",
    "tbill_3m":     f"{BASE}/other/3m_yield_processed.csv",

    # Inflation
    "cpi":          f"{BASE}/inflation/cpi_processed.csv",
    "pce":          f"{BASE}/inflation/PCE_price_index_processed.csv",
    "ppi":          f"{BASE}/inflation/PPI_inflation_processed.csv",

    # Growth
    "indprod":      f"{BASE}/ec_growth/industrial_production_processed.csv",
    "retail_sales": f"{BASE}/ec_growth/retail_sales_processed.csv",
    "inventories":  f"{BASE}/ec_growth/tot_business_inventories_processed.csv",
    "export_px":    f"{BASE}/ec_growth/export_price_index_processed.csv",
    "import_px":    f"{BASE}/ec_growth/import_price_index_processed.csv",
    "unemp":        f"{BASE}/ec_growth/unemployment_processed.csv",

    # Money & policy
    "m2":           f"{BASE}/mon_policy/m2_real_money_supply_processed.csv",
    "fedfunds":     f"{BASE}/mon_policy/fedfunds_processed.csv",
    "discount_rate":f"{BASE}/mon_policy/fed_reserve_discount_rate_processed.csv",

    # Yields & spreads
    "spread_10y_2y":f"{BASE}/mkt_vol/10y_2y_spread_processed.csv",
    "nat_fin_cond": f"{BASE}/mkt_vol/nat_fin_condition_indx_processed.csv",
    "nasdaq_vol":   f"{BASE}/mkt_vol/nasdaq_vol_indx_processed.csv",
    "hy_spread":    f"{BASE}/other/bofa_highyield_spread_processed.csv",
    "y2":           f"{BASE}/other/2y_yield_processed.csv",
    "y3m":          f"{BASE}/other/3m_yield_processed.csv",
    "y10":          f"{BASE}/other/10y_yield_processed.csv",
}


# ============================================================
# STEP 1 — BUILD EMRP VIA GEOMETRIC RETURNS
# ============================================================

# --------- S&P 500 geometric monthly return ---------
sp = pd.read_csv(paths["sp500"], parse_dates=["date"]).set_index("date")
sp = sp.sort_index()
sp["sp500_geo"] = sp["pct_change_mom"] / 100.0     # monthly %
# Already monthly, but ensure month-end alignment
sp_m = sp.resample("M").last()[["sp500_geo"]]

# --------- 3M T-bill geometric monthly return ---------
rf = pd.read_csv(paths["tbill_3m"], parse_dates=["date"]).set_index("date")
rf = rf.sort_index()

# daily annualized % yield → daily simple return
rf["r_daily"] = (rf["value"] / 100) / 252

# geometric compounding inside each month
rf_m = rf["r_daily"].resample("M").apply(lambda x: (1 + x).prod() - 1)
rf_m = rf_m.to_frame(name="rf_geo")

# --------- EMRP ---------
df = sp_m.join(rf_m, how="inner")
df["EMRP_geo"] = df["sp500_geo"] - df["rf_geo"]
df["EMRP_next"] = df["EMRP_geo"].shift(-1)   # predict next month's excess return


# ============================================================
# STEP 2 — BUILD MACRO PREDICTORS (MONTH-END)
# ============================================================

def monthly_last(path, date_col="date", value_col="value", new_name=None):
    df = pd.read_csv(path, parse_dates=[date_col])
    df = df[[date_col, value_col]].set_index(date_col).sort_index()
    m = df.resample("M").last()
    return m.rename(columns={value_col: new_name})

# YoY series
cpi = monthly_last(paths["cpi"], value_col="pct_change_yoy", new_name="cpi_yoy")
pce = monthly_last(paths["pce"], value_col="pct_change_yoy", new_name="pce_yoy")
ppi = monthly_last(paths["ppi"], value_col="pct_change_yoy", new_name="ppi_yoy")
indprod = monthly_last(paths["indprod"], value_col="pct_change_yoy", new_name="indprod_yoy")
retail = monthly_last(paths["retail_sales"], value_col="pct_change_yoy", new_name="retail_sales_yoy")
invent = monthly_last(paths["inventories"], value_col="pct_change_yoy", new_name="inventories_yoy")
export_px = monthly_last(paths["export_px"], value_col="pct_change_yoy", new_name="export_px_yoy")
import_px = monthly_last(paths["import_px"], value_col="pct_change_yoy", new_name="import_px_yoy")
m2 = monthly_last(paths["m2"], value_col="pct_change_yoy", new_name="m2_yoy")

# Levels
unemp = monthly_last(paths["unemp"], value_col="value", new_name="unemp_rate")
fedfunds = monthly_last(paths["fedfunds"], value_col="value", new_name="fedfunds")
discount_rate = monthly_last(paths["discount_rate"], value_col="value", new_name="discount_rate")
spread = monthly_last(paths["spread_10y_2y"], value_col="value", new_name="spread_10y_2y")
nat_fin = monthly_last(paths["nat_fin_cond"], value_col="value", new_name="nat_fin_cond")
nasdaq_vol = monthly_last(paths["nasdaq_vol"], value_col="value", new_name="nasdaq_vol")
hy = monthly_last(paths["hy_spread"], value_col="value", new_name="hy_spread")
y2 = monthly_last(paths["y2"], value_col="value", new_name="y2")
y3m = monthly_last(paths["y3m"], value_col="value", new_name="y3m")
y10 = monthly_last(paths["y10"], value_col="value", new_name="y10")

macro = pd.concat([
    cpi, pce, ppi, indprod, retail, invent,
    export_px, import_px,
    unemp, m2, fedfunds, discount_rate,
    spread, nat_fin, nasdaq_vol, hy, y2, y3m, y10
], axis=1)

# Final merged dataset
reg_df = df.join(macro, how="inner").dropna()


# ============================================================
# STEP 3 — MULTIVARIATE OLS
# ============================================================

predictors = list(macro.columns)

df_multi = reg_df[["EMRP_next"] + predictors].dropna()

y = df_multi["EMRP_next"]
X = sm.add_constant(df_multi[predictors])

model_multi = sm.OLS(y, X).fit()

print("\n\n=== MULTIPLE OLS REGRESSION ===")
print(model_multi.summary())


# ============================================================
# STEP 4 — UNIVARIATE (SIMPLE) OLS FOR EACH VARIABLE
# ============================================================

simple_rows = []

for var in predictors:
    tmp = reg_df[["EMRP_next", var]].dropna()

    y_s = tmp["EMRP_next"]
    X_s = sm.add_constant(tmp[[var]])

    res = sm.OLS(y_s, X_s).fit()

    simple_rows.append({
        "variable": var,
        "coef": res.params[var],
        "t_stat": res.tvalues[var],
        "p_value": res.pvalues[var],
        "R_squared": res.rsquared,
        "n_obs": int(res.nobs),
        "sign": "positive" if res.params[var] > 0 else "negative"
    })

simple_df = pd.DataFrame(simple_rows).sort_values("p_value")
print("\n\n=== SIMPLE UNIVARIATE RESULTS ===")
print(simple_df.to_string(index=False))


# ============================================================
# STEP 5 — LASSO SELECTION + REDUCED OLS
# ============================================================

X_full = reg_df[predictors].values
y_full = reg_df["EMRP_next"].values

lasso_pipe = Pipeline([
    ("scaler", StandardScaler()),
    ("lasso", LassoCV(cv=5, random_state=0))
])

lasso_pipe.fit(X_full, y_full)

lasso = lasso_pipe.named_steps["lasso"]
lasso_alpha = lasso.alpha_
coefs = lasso.coef_

selected = [p for p, c in zip(predictors, coefs) if abs(c) > 1e-6]

print("\n\n=== LASSO RESULTS ===")
print("Chosen alpha:", lasso_alpha)
print("Selected variables:", selected)

# Reduced OLS
df_reduced = reg_df[["EMRP_next"] + selected].dropna()
y_r = df_reduced["EMRP_next"]
X_r = sm.add_constant(df_reduced[selected])

model_reduced = sm.OLS(y_r, X_r).fit()

print("\n\n=== REDUCED MULTIVARIATE OLS (POST-LASSO) ===")
print(model_reduced.summary())

/var/folders/h_/5wg7lw3n2djc8yh6g107mytm0000gn/T/ipykernel_46673/775581645.py:57: FutureWarning: 'M' is deprecated and will be removed in a future version, please use 'ME' instead.
  sp_m = sp.resample("M").last()[["sp500_geo"]]
/var/folders/h_/5wg7lw3n2djc8yh6g107mytm0000gn/T/ipykernel_46673/775581645.py:67: FutureWarning: 'M' is deprecated and will be removed in a future version, please use 'ME' instead.
  rf_m = rf["r_daily"].resample("M").apply(lambda x: (1 + x).prod() - 1)
/var/folders/h_/5wg7lw3n2djc8yh6g107mytm0000gn/T/ipykernel_46673/775581645.py:83: FutureWarning: 'M' is deprecated and will be removed in a future version, please use 'ME' instead.
  m = df.resample("M").last()
/var/folders/h_/5wg7lw3n2djc8yh6g107mytm0000gn/T/ipykernel_46673/775581645.py:83: FutureWarning: 'M' is deprecated and will be removed in a future version, please use 'ME' instead.
  m = df.resample("M").last()
/var/folders/h_/5wg7lw3n2djc8yh6g107mytm0000gn/T/ipykernel_46673/775581645.py:83: FutureWarning



=== MULTIPLE OLS REGRESSION ===
                            OLS Regression Results                            
Dep. Variable:              EMRP_next   R-squared:                       0.181
Model:                            OLS   Adj. R-squared:                  0.116
Method:                 Least Squares   F-statistic:                     2.797
Date:                Mon, 24 Nov 2025   Prob (F-statistic):           0.000208
Time:                        01:50:09   Log-Likelihood:                 452.41
No. Observations:                 247   AIC:                            -866.8
Df Residuals:                     228   BIC:                            -800.2
Df Model:                          18                                         
Covariance Type:            nonrobust                                         
                       coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------------
const 

Quarterly  

In [3]:
import pandas as pd
import numpy as np
import statsmodels.api as sm

from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Lasso
from sklearn.pipeline import Pipeline

# ============================================================
# STEP 0 — Paths
# ============================================================

BASE = "data/macro_processed"

paths = {
    # Returns building blocks
    "sp500":        f"{BASE}/other/sp500_processed.csv",
    "tbill_3m":     f"{BASE}/other/3m_yield_processed.csv",

    # Inflation
    "cpi":          f"{BASE}/inflation/cpi_processed.csv",
    "pce":          f"{BASE}/inflation/PCE_price_index_processed.csv",
    "ppi":          f"{BASE}/inflation/PPI_inflation_processed.csv",

    # Growth
    "indprod":      f"{BASE}/ec_growth/industrial_production_processed.csv",
    "retail_sales": f"{BASE}/ec_growth/retail_sales_processed.csv",
    "inventories":  f"{BASE}/ec_growth/tot_business_inventories_processed.csv",
    "export_px":    f"{BASE}/ec_growth/export_price_index_processed.csv",
    "import_px":    f"{BASE}/ec_growth/import_price_index_processed.csv",
    "unemp":        f"{BASE}/ec_growth/unemployment_processed.csv",

    # Money and policy
    "m2":           f"{BASE}/mon_policy/m2_real_money_supply_processed.csv",
    "fedfunds":     f"{BASE}/mon_policy/fedfunds_processed.csv",
    "discount_rate":f"{BASE}/mon_policy/fed_reserve_discount_rate_processed.csv",

    # Yields and spreads
    "spread_10y_2y":f"{BASE}/mkt_vol/10y_2y_spread_processed.csv",
    "nat_fin_cond": f"{BASE}/mkt_vol/nat_fin_condition_indx_processed.csv",
    "nasdaq_vol":   f"{BASE}/mkt_vol/nasdaq_vol_indx_processed.csv",
    "hy_spread":    f"{BASE}/other/bofa_highyield_spread_processed.csv",
    "y2":           f"{BASE}/other/2y_yield_processed.csv",
    "y3m":          f"{BASE}/other/3m_yield_processed.csv",
    "y10":          f"{BASE}/other/10y_yield_processed.csv",
}


# ============================================================
# STEP 1 — Monthly geometric EMRP, then quarterly aggregation
# ============================================================

# ----- S&P 500: monthly geometric return -----
sp = pd.read_csv(paths["sp500"], parse_dates=["date"]).set_index("date")
sp = sp.sort_index()
sp["sp500_geo_m"] = sp["pct_change_mom"] / 100.0                # already monthly %
sp_m = sp.resample("M").last()[["sp500_geo_m"]]                 # make sure month end

# ----- 3m T-bill: daily -> monthly geometric return -----
rf = pd.read_csv(paths["tbill_3m"], parse_dates=["date"]).set_index("date")
rf = rf.sort_index()
rf["r_daily"] = (rf["value"] / 100.0) / 252.0                   # simple daily

rf_m = rf["r_daily"].resample("M").apply(lambda x: (1 + x).prod() - 1)
rf_m = rf_m.to_frame(name="rf_geo_m")

# ----- Monthly EMRP, then quarterly geometric aggregation -----
m = sp_m.join(rf_m, how="inner")
m["EMRP_geo_m"] = m["sp500_geo_m"] - m["rf_geo_m"]

# quarterly geometric EMRP (compounding monthly EMRP)
emrp_q = (1 + m["EMRP_geo_m"]).resample("Q").prod() - 1
emrp_q = emrp_q.to_frame(name="EMRP_q")
emrp_q["EMRP_next_q"] = emrp_q["EMRP_q"].shift(-1)


# ============================================================
# STEP 2 — Macro predictors, quarterly (last month of each quarter)
# ============================================================

def monthly_last(path, date_col="date", value_col="value", new_name=None):
    df = pd.read_csv(path, parse_dates=[date_col])
    df = df[[date_col, value_col]].set_index(date_col).sort_index()
    m = df.resample("M").last()
    if new_name is None:
        new_name = value_col
    return m.rename(columns={value_col: new_name})

# YoY transforms
cpi   = monthly_last(paths["cpi"],   value_col="pct_change_yoy", new_name="cpi_yoy")
pce   = monthly_last(paths["pce"],   value_col="pct_change_yoy", new_name="pce_yoy")
ppi   = monthly_last(paths["ppi"],   value_col="pct_change_yoy", new_name="ppi_yoy")
ind   = monthly_last(paths["indprod"],      value_col="pct_change_yoy", new_name="indprod_yoy")
ret   = monthly_last(paths["retail_sales"], value_col="pct_change_yoy", new_name="retail_sales_yoy")
inv   = monthly_last(paths["inventories"],  value_col="pct_change_yoy", new_name="inventories_yoy")
exp_px = monthly_last(paths["export_px"],   value_col="pct_change_yoy", new_name="export_px_yoy")
imp_px = monthly_last(paths["import_px"],   value_col="pct_change_yoy", new_name="import_px_yoy")
m2    = monthly_last(paths["m2"],    value_col="pct_change_yoy", new_name="m2_yoy")

# Level series
unemp   = monthly_last(paths["unemp"],         value_col="value", new_name="unemp_rate")
fedf    = monthly_last(paths["fedfunds"],      value_col="value", new_name="fedfunds")
disc    = monthly_last(paths["discount_rate"], value_col="value", new_name="discount_rate")
spread  = monthly_last(paths["spread_10y_2y"], value_col="value", new_name="spread_10y_2y")
nat_fin = monthly_last(paths["nat_fin_cond"],  value_col="value", new_name="nat_fin_cond")
nasd_v  = monthly_last(paths["nasdaq_vol"],    value_col="value", new_name="nasdaq_vol")
hy      = monthly_last(paths["hy_spread"],     value_col="value", new_name="hy_spread")
y2      = monthly_last(paths["y2"],            value_col="value", new_name="y2")
y3m     = monthly_last(paths["y3m"],           value_col="value", new_name="y3m")
y10     = monthly_last(paths["y10"],           value_col="value", new_name="y10")

macro_m = pd.concat([
    cpi, pce, ppi,
    ind, ret, inv,
    exp_px, imp_px,
    unemp, m2, fedf, disc,
    spread, nat_fin, nasd_v, hy,
    y2, y3m, y10
], axis=1)

# convert to quarter: last month of each quarter
macro_q = macro_m.resample("Q").last()

# ============================================================
# STEP 3 — Merge quarterly EMRP and macro series
# ============================================================

reg_q = emrp_q.join(macro_q, how="inner").dropna()

predictors = list(macro_q.columns)

print("Quarterly sample size:", len(reg_q))

# ============================================================
# STEP 4 — Full multivariate quarterly OLS
# ============================================================

df_multi_q = reg_q[["EMRP_next_q"] + predictors].dropna()
y_q = df_multi_q["EMRP_next_q"]
X_q = sm.add_constant(df_multi_q[predictors])

model_q = sm.OLS(y_q, X_q).fit()

print("\n=== QUARTERLY MULTIPLE OLS REGRESSION ===")
print(model_q.summary())


# ============================================================
# STEP 5 — Univariate quarterly OLS for each predictor
# ============================================================

simple_rows_q = []

for var in predictors:
    tmp = reg_q[["EMRP_next_q", var]].dropna()
    y_s = tmp["EMRP_next_q"]
    X_s = sm.add_constant(tmp[[var]])
    res = sm.OLS(y_s, X_s).fit()

    simple_rows_q.append({
        "variable": var,
        "coef": res.params[var],
        "t_stat": res.tvalues[var],
        "p_value": res.pvalues[var],
        "R_squared": res.rsquared,
        "n_obs": int(res.nobs),
        "sign": "positive" if res.params[var] > 0 else "negative"
    })

simple_q_df = pd.DataFrame(simple_rows_q).sort_values("p_value")
print("\n=== QUARTERLY SIMPLE UNIVARIATE RESULTS ===")
print(simple_q_df.to_string(index=False))


# ============================================================
# STEP 6 — LASSO with controlled sparsity (3–6 variables)
# ============================================================

X_full_q = reg_q[predictors].values
y_full_q = reg_q["EMRP_next_q"].values

alphas = np.logspace(-3, 0, 60)  # from 0.001 to 1
tol = 1e-6

best_alpha = None
best_coefs = None
best_k = None

# loop from large alpha (strong penalty) down to small (weak penalty)
for a in sorted(alphas, reverse=True):
    pipe = Pipeline([
        ("scaler", StandardScaler()),
        ("lasso", Lasso(alpha=a, max_iter=20000, random_state=0))
    ])
    pipe.fit(X_full_q, y_full_q)
    coefs = pipe.named_steps["lasso"].coef_
    k = np.sum(np.abs(coefs) > tol)
    if 3 <= k <= 6:
        best_alpha = a
        best_coefs = coefs
        best_k = k
        break

# fallback: if we never hit 3–6, take the smallest alpha (weakest penalty)
if best_alpha is None:
    a = alphas.min()
    pipe = Pipeline([
        ("scaler", StandardScaler()),
        ("lasso", Lasso(alpha=a, max_iter=20000, random_state=0))
    ])
    pipe.fit(X_full_q, y_full_q)
    best_alpha = a
    best_coefs = pipe.named_steps["lasso"].coef_
    best_k = np.sum(np.abs(best_coefs) > tol)

selected_vars = [p for p, c in zip(predictors, best_coefs) if abs(c) > tol]

print("\n=== QUARTERLY LASSO RESULTS (FORCED 3–6 VARIABLES) ===")
print("Chosen alpha:", best_alpha)
print("Number of nonzero coefficients:", int(best_k))
print("Selected variables:", selected_vars)

print("\nLasso coefficients (standardised scale):")
for p, c in sorted(zip(predictors, best_coefs), key=lambda x: -abs(x[1])):
    if abs(c) > tol:
        print(f"{p:18s} {c:+.5f}")


# ============================================================
# STEP 7 — Reduced quarterly OLS on LASSO-selected set
# ============================================================

df_red_q = reg_q[["EMRP_next_q"] + selected_vars].dropna()
y_rq = df_red_q["EMRP_next_q"]
X_rq = sm.add_constant(df_red_q[selected_vars])

model_red_q = sm.OLS(y_rq, X_rq).fit()

print("\n=== QUARTERLY REDUCED MULTIVARIATE OLS (POST LASSO) ===")
print(model_red_q.summary())

/var/folders/h_/5wg7lw3n2djc8yh6g107mytm0000gn/T/ipykernel_92106/2671417733.py:57: FutureWarning: 'M' is deprecated and will be removed in a future version, please use 'ME' instead.
  sp_m = sp.resample("M").last()[["sp500_geo_m"]]                 # make sure month end
/var/folders/h_/5wg7lw3n2djc8yh6g107mytm0000gn/T/ipykernel_92106/2671417733.py:64: FutureWarning: 'M' is deprecated and will be removed in a future version, please use 'ME' instead.
  rf_m = rf["r_daily"].resample("M").apply(lambda x: (1 + x).prod() - 1)
/var/folders/h_/5wg7lw3n2djc8yh6g107mytm0000gn/T/ipykernel_92106/2671417733.py:72: FutureWarning: 'Q' is deprecated and will be removed in a future version, please use 'QE' instead.
  emrp_q = (1 + m["EMRP_geo_m"]).resample("Q").prod() - 1
/var/folders/h_/5wg7lw3n2djc8yh6g107mytm0000gn/T/ipykernel_92106/2671417733.py:84: FutureWarning: 'M' is deprecated and will be removed in a future version, please use 'ME' instead.
  m = df.resample("M").last()
/var/folders/h_/5wg7lw3

Quarterly sample size: 83

=== QUARTERLY MULTIPLE OLS REGRESSION ===
                            OLS Regression Results                            
Dep. Variable:            EMRP_next_q   R-squared:                       0.459
Model:                            OLS   Adj. R-squared:                  0.306
Method:                 Least Squares   F-statistic:                     3.013
Date:                Mon, 24 Nov 2025   Prob (F-statistic):           0.000611
Time:                        22:07:01   Log-Likelihood:                 114.44
No. Observations:                  83   AIC:                            -190.9
Df Residuals:                      64   BIC:                            -144.9
Df Model:                          18                                         
Covariance Type:            nonrobust                                         
                       coef    std err          t      P>|t|      [0.025      0.975]
--------------------------------------------------------

/var/folders/h_/5wg7lw3n2djc8yh6g107mytm0000gn/T/ipykernel_92106/2671417733.py:84: FutureWarning: 'M' is deprecated and will be removed in a future version, please use 'ME' instead.
  m = df.resample("M").last()
/var/folders/h_/5wg7lw3n2djc8yh6g107mytm0000gn/T/ipykernel_92106/2671417733.py:84: FutureWarning: 'M' is deprecated and will be removed in a future version, please use 'ME' instead.
  m = df.resample("M").last()
/var/folders/h_/5wg7lw3n2djc8yh6g107mytm0000gn/T/ipykernel_92106/2671417733.py:122: FutureWarning: 'Q' is deprecated and will be removed in a future version, please use 'QE' instead.
  macro_q = macro_m.resample("Q").last()
